#### Model will be Trained on One Attribute for Testing

##### Imports

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

import catboost as cb
from catboost import CatBoostRegressor

##### Directories and File Locations

In [ ]:
BASE_DIR = Path.cwd().parent
SPLITS_DIR = BASE_DIR / "data" / "splits_2"

In [6]:
train_df = pd.read_csv(SPLITS_DIR / "train.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

Features that were selected from the previous notebook

In [7]:
feature_cols = ['GP_base', 
                'MIN_base', 
                'FG_PCT_base', 
                'FG3M', 
                'FG3A', 
                'FG3_PCT', 
                'FTM', 
                'FTA', 
                'FT_PCT', 
                'OREB', 
                'DREB', 
                'REB', 
                'AST', 
                'TOV', 
                'STL', 
                'BLK', 
                'BLKA', 
                'PF', 
                'PFD', 
                'PTS', 
                'PLUS_MINUS', 
                'OFF_RATING', 
                'DEF_RATING', 
                'NET_RATING', 
                'AST_PCT', 
                'AST_TO', 
                'AST_RATIO', 
                'OREB_PCT', 
                'DREB_PCT', 
                'REB_PCT', 
                'E_TOV_PCT', 
                'EFG_PCT', 
                'TS_PCT', 
                'USG_PCT', 
                'PACE', 
                'PIE', 
                'POSS', 
                'FGM_PG', 
                'FGA_PG']

Labels

In [8]:
target_cols = [
    "overallAttribute",
    "closeShot",
    "midRangeShot",
    "threePointShot",
    "freeThrow",
    "shotIQ",
    "offensiveConsistency",
    "layup",
    "standingDunk",
    "drivingDunk",
    "postHook",
    "postFade",
    "postControl",
    "drawFoul",
    "hands",
    "interiorDefense",
    "perimeterDefense",
    "steal",
    "block",
    "helpDefenseIQ",
    "passPerception",
    "defensiveConsistency",
    "speed",
    "strength",
    "vertical",
    "stamina",
    "hustle",
    "overallDurability",
    "passAccuracy",
    "ballHandle",
    "speedWithBall",
    "passIQ",
    "passVision",
    "offensiveRebound",
    "defensiveRebound",
    "agility"
]

Create X and Y

In [9]:
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

##### CatBoost

In [10]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score

results = []

for target_col in target_cols:

    Y_train = train_df[target_col]
    Y_test = test_df[target_col]

    model = CatBoostRegressor(
        iterations=2000,
        depth=6,
        learning_rate=0.03,
        loss_function='RMSE',
        random_state=42,
        verbose=0
    )

    model.fit(
        X_train,
        Y_train,
        eval_set=(X_test, Y_test),
        use_best_model=True
    )

    preds = model.predict(X_test)

    mae = mean_absolute_error(Y_test, preds)
    r2 = r2_score(Y_test, preds)

    results.append({
        "Attribute": target_col,
        "MAE": round(mae, 2),
        "R2": round(r2, 3)
    })

##### Print Results

In [ ]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="R2",
    ascending=True
)

,Attribute,MAE,R2
33,offensiveRebound,3.64,0.870
32,passVision,3.50,0.860
0,overallAttribute,1.78,0.858
34,defensiveRebound,3.01,0.855
18,block,4.83,0.796
3,threePointShot,3.47,0.767
4,freeThrow,3.70,0.672
29,ballHandle,6.36,0.657
8,standingDunk,9.38,0.639
17,steal,6.30,0.632
